# Orbital-resolved quantum geometric tensor: 2D Haldane, benzene and coronene ($t'=1.0\,t$)

This is `Quantum_Metric_Wannier.ipynb`, adapted to run on the PythTB models of `bond_index_benzene_chern.ipynb` instead of a Wannier90 model. It keeps the original workflow: hopping table, then `ham_func`, then `TBLR.TBLinearResponse`, then the orbital-resolved QGT.

**Outputs, written to `OUT/<system>/`:**
* `qgt_xx.npy`, `qgt_xy.npy`, `qgt_yx.npy`, `qgt_yy.npy`: the four-index tensors of the original `calc_QM`, with index order `Q[a,b,c,d] = <o|v_ad|u><u|v_bc|o>/(E_u-E_o)^2`, normalized per electron.
* `fluctuation_quantum_metric.csv` and `fluctuation_berry_curvature.csv`: the last cells of `Quantum_Metric_Wannier.ipynb`, with CSV in place of the xlsx. Metric: `qgtxx + qgtyy`. Berry curvature: `i(qgtxy - qgtyx)`, whose total is $-2\,\mathrm{Im}\,\mathcal Q^{xy}$. Patterns label the `qgt[a,b,c,d]` indices, and the real part is kept, as in the original `np.float64` cast.
* `berry_loops.csv` and `berry_loop_families.csv`: the orbital-resolved Berry curvature as three-centre bond loops, as in `bond_index_benzene_chern.ipynb`, with the `einsum` index fix.
* `summary.csv`: totals and cross-checks.

**Changes from the original notebook**
1. 2D only: 2-component k, 2×2 lattice, 2-component hopping vectors, and `Nkp=[nk, nk]`. Molecules use `Nkp=[1, 1]`.
2. `calc_QM` also returns `qgtxy` and `qgtyx`, and uses a central difference for $\partial H/\partial k$, which is still built from `tb.ham`.
3. Fixed the save cell, which wrote `qgtxx` into the `qgtyx` file, and the missing `/` in the file paths.
4. The loops take $\rho(\mathbf R)$ from `tb.solve_1pRDM_kresolved`. The geometry factor uses $(\mathbf r_a-\mathbf r_b)\times(\mathbf r_a-\mathbf r_c)$; the old `einsum("ba,ca->abc", x_ba, …)` read `x_ba` transposed.
5. Loop prefactor: $C$ contribution $=(2\pi/A_{\rm cell})\,(\mathbf r_a-\mathbf r_b)\times(\mathbf r_a-\mathbf r_c)\,\mathrm{Im}\,b_{abc}$, using the honeycomb cell for the molecules too. The old prefactor $2\pi N_{\rm occ}/A_{\rm supercell}$ rescaled molecule loops by 3/4.

**Supercell.** `haldane_4x4` is the same Haldane model on a 4×4 supercell with 32 sites, built with `make_supercell`. Each site is then its own orbital index in the four-index tensor and the fluctuation tables; hops that cross the supercell boundary are still folded onto the 32 indices. Its k mesh is `NK/4`, the same k-point density as the primitive cell, so the totals must agree to mesh accuracy. Its loop table uses `rmax = ceil(RMAX/4)` supercells and writes only loops with |(r_a−r_b)×(r_a−r_c) Im b| > `LOOP_TOL_SC`, and the primitive-cell table uses `LOOP_TOL`. `summary.csv` reports the loop sum both before and after the cut.

**Conventions.** Lengths are in units of the nearest-neighbour bond $a$; energies are in $|t|$. `PHI = -pi/2` is the manuscript convention ($C=+1$); `+pi/2` is the original notebook ($C=-1$). For the 2-orbital Haldane cell, an "orbital" is a sublattice summed over all its lattice images, as in the Wannier workflow. The loop table resolves the images explicitly.

**Note on `TBLR.solve_QGT_kresolved_offD`.** It sums `pref*|Jx|*|Jy|`, which is neither the real nor the imaginary part of $\mathcal Q^{xy}$. It cannot give the Berry curvature, so this notebook does not use it.


In [1]:
#Import everything
import numpy as np, pandas as pd, os, json, itertools
import matplotlib.pyplot as plt
from tqdm import tqdm
from pythtb import Lattice, TBModel
import TBLR2 as TBLR
try:
    import opt_einsum as oe
    contract = oe.contract
except ImportError:                                   # opt_einsum is optional
    contract = lambda *a: np.einsum(*a, optimize=True)

# ---------------- parameters
T1      = -1.0          # nearest-neighbour hopping
T2_MAG  = 1.0          # |t'|/|t|
PHI     = -np.pi/2      # -pi/2: manuscript convention (C=+1); +pi/2: original notebook (C=-1)
DELTA   = 0.0           # sublattice mass
NK      = 100           # periodic model: NK x NK k mesh
RMAX    = 12             # periodic model: lattice-image box for the three-centre loops
MU      = 0.0           # chemical potential (half filling)
BETA    = 1.e2          # Fermi smearing, as in the original notebook
K_STEP  = 1.e-4         # finite-difference step for dH/dk (central difference)
ACC     = 1/np.sqrt(3)  # NN bond in PythTB lattice units -> lengths are reported in units of a
SC      = 4             # Haldane supercell size (SC x SC); its k mesh is NK//SC so it samples the same k-points
LOOP_TOL = 1e-8       # primitive-cell loop table: keep loops with |(r_a-r_b)x(r_a-r_c) Im b| > tol
LOOP_TOL_SC = 1e-5      # supercell loop table: keep loops with |(r_a-r_b)x(r_a-r_c) Im b| > tol (7.8M -> 0.12M rows; full sum still reported)
OUT     = "qgt_orbital_resolved_tprime_1p0"

In [2]:
#Models: same hoppings as bond_index_benzene_chern.ipynb
def build_models(t2mag=T2_MAG, phi=PHI, delta=DELTA, t=T1):
    lat = Lattice([[1, 0], [0.5, np.sqrt(3)/2]], [[1/3, 1/3], [2/3, 2/3]], periodic_dirs=...)
    h = TBModel(lat)
    t2 = t2mag*np.exp(1j*phi); t2c = np.conj(t2)
    h.set_onsite([-delta, delta])
    h.set_hop(t, 0, 1, [0, 0]); h.set_hop(t, 1, 0, [1, 0]); h.set_hop(t, 1, 0, [0, 1])
    h.set_hop(t2, 0, 0, [1, 0]); h.set_hop(t2, 1, 1, [1, -1]); h.set_hop(t2, 1, 1, [0, 1])
    h.set_hop(t2c, 1, 1, [1, 0]); h.set_hop(t2c, 0, 0, [1, -1]); h.set_hop(t2c, 0, 0, [0, 1])
    s1 = h.make_supercell(np.array([[2, 0], [0, 2]]))
    bz = s1.cut_piece(1, 1, glue_edges=False).cut_piece(1, 0, glue_edges=False)
    bz.remove_orb(7); bz.remove_orb(0)
    s2 = h.make_supercell(np.array([[4, 0], [0, 4]]))
    co = s2.cut_piece(1, 1, glue_edges=False).cut_piece(1, 0, glue_edges=False)
    for o in [31, 30, 29, 23, 8, 2, 1, 0]: co.remove_orb(o)
    hsc = h.make_supercell(np.array([[SC, 0], [0, SC]]))          # SC x SC supercell of the periodic model
    return {"haldane": h, f"haldane_{SC}x{SC}": hsc, "benzene": bz, "coronene": co}

def is_molecule(model):
    return not any(np.any(np.asarray(hp.get("lattice_vector", [0, 0])) != 0) for hp in model.hoppings)

In [3]:
#Create array with all hopping information (2D)
def hopping_table(my_model):
    norb = my_model.norb
    H = np.empty((norb, norb), dtype=object)
    for i in range(norb):
        for j in range(norb):
            H[i, j] = []
    orb_pos = np.asarray(my_model.orb_vecs)
    lat = np.asarray(my_model.get_lat_vecs())/ACC          # lengths in units of a
    for hop in my_model.hoppings:
        t = hop['amplitude']
        i = hop['from_orbital']
        j = hop['to_orbital']
        j_image = np.asarray(hop.get('lattice_vector', [0, 0]))
        dist = ((orb_pos[j]+j_image)-orb_pos[i]) @ lat
        H[i, j].append([t, dist[0], dist[1]])
    return H, lat, orb_pos @ lat

#Function that finds hamiltonian from hopping array at specific k-point (2D)
def make_ham_func(H, my_model):
    norb = my_model.norb
    onsite = np.asarray(my_model._site_energies, float)
    def ham_func(k):
        kx, ky = k
        H_res = np.zeros((norb, norb), dtype=complex)
        for i in range(norb):
            for j in range(norb):
                for item in H[i][j]:
                    v = np.round(complex(item[0]), 10)*np.exp(1j*kx*item[1]+1j*ky*item[2])
                    H_res[i][j] += v
                    H_res[j][i] += np.conj(v)
        for i, e in enumerate(onsite):
            H_res[i, i] += np.round(e, 10)
        return H_res
    return ham_func

In [4]:
#Orbital-resolved QGT (original calc_QM, 2D, now with xy and yx), per electron
def Fermi(beta_const, x):
    return 1.0/(1.0 + np.exp(beta_const*x))

def FNM(em, en, beta_const=BETA, delta=1.e-3):
    emn = em - en
    fnm = Fermi(beta_const, en)*(1.0 - Fermi(beta_const, em))
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.abs(emn) < delta, 0.0, fnm/(emn**2))

def calc_QM(tb, occ, mu=MU, k_step=K_STEP):
    dkx, dky = np.array([1, 0]), np.array([0, 1])
    e = tb.enUM[:, :, 0].real - mu
    pref = FNM(e[:, np.newaxis, :], e[:, :, np.newaxis])     # pref[k,i,j] = f_i(1-f_j)/(E_j-E_i)^2
    n = tb.norb
    Q = {c: np.zeros((n, n, n, n), complex) for c in ("xx", "xy", "yx", "yy")}
    for i, k in enumerate(tqdm(tb.kSpan, disable=len(tb.kSpan) < 2)):
        um = tb.enUM[i, :, 1:]
        uH = um.conj().T
        dH = {"x": (tb.ham(k + k_step*dkx) - tb.ham(k - k_step*dkx))/(2*k_step),
              "y": (tb.ham(k + k_step*dky) - tb.ham(k - k_step*dky))/(2*k_step)}
        Jv1 = {c: contract("ia,ad,dj->adij", uH, dH[c], um) for c in "xy"}             # <i|v_ad|j>
        Jv2c = {c: np.conj(contract("ic,cb,bj->bcij", uH, dH[c], um)) for c in "xy"}   # <j|v_bc|i>
        for c in Q:
            Q[c] += contract("ij,adij,bcij->abcd", pref[i], Jv1[c[0]], Jv2c[c[1]])
    denom = len(tb.kSpan)*occ
    return {c: Q[c]/denom for c in Q}

In [5]:
#Fluctuation-resolved quantum metric and Berry curvature (last cells of Quantum_Metric_Wannier.ipynb)
import itertools

def canonicalize(combo):
    mapping = {}
    next_label = ord('a')
    result = []
    for x in combo:
        if x not in mapping:
            mapping[x] = chr(next_label)
            next_label += 1
        result.append(mapping[x])
    return ''.join(result)

def find_combo_list(orb_list, num_un):
    patterns = set()
    for combo in itertools.product(orb_list, repeat=4):
        if len(set(combo)) >= num_un:
            patterns.add(canonicalize(combo))
    return sorted(patterns)

def find_atom_metric(atomic_metric, num_un, atoms):
    atomic_metric_ab = atomic_metric.copy()
    for a in range(len(atoms)):
        for b in range(len(atoms)):
            for c in range(len(atoms)):
                indices = [a, b, c][:num_un]
                direct_indices = np.asarray(list(itertools.product(indices, repeat=4)))
                for ind in direct_indices:
                    atomic_metric_ab[ind[0], ind[1], ind[2], ind[3]] = 0
    return atomic_metric_ab

combo_a = find_combo_list(["a"], 1)
combo_ab = find_combo_list(["a", "b"], 2)
combo_abc = find_combo_list(["a", "b", "c"], 3)
combo_abcd = find_combo_list(["a", "b", "c", "d"], 4)

def fluctuation_table(tensor, atoms, label, denom=1):
    """tensor[a,b,c,d] in the qgt index order; returns the 1orb/2orb/3orb/4orb table of the original notebook."""
    atomic = np.zeros((len(atoms),)*4, dtype=complex)
    for i, m in enumerate(atoms):
        for j, n in enumerate(atoms):
            for k, o in enumerate(atoms):
                for l, p in enumerate(atoms):
                    atomic[i, j, k, l] = np.sum(tensor[np.ix_(m, n, o, p)])
    atom_metric_ab = find_atom_metric(atomic, 1, atoms)
    atom_metric_abc = find_atom_metric(atomic, 2, atoms)
    atom_metric_abcd = find_atom_metric(atomic, 3, atoms)
    data = [[], [], [], []]; dropped_imag = 0.0
    sets = [(combo_a, atomic), (np.setdiff1d(combo_ab, combo_a), atom_metric_ab),
            (np.setdiff1d(combo_abc, combo_ab), atom_metric_abc), (np.setdiff1d(combo_abcd, combo_abc), atom_metric_abcd)]
    for n, (combos, arr) in enumerate(sets):
        for combo in combos:
            v = np.einsum(str(combo)+"->", arr)/denom
            dropped_imag = max(dropped_imag, abs(v.imag))
            data[n].append([str(combo), np.float64(v.real)])     # the original keeps the real part (np.float64 cast)
    dfs = [pd.DataFrame(data=np.array(d, dtype=object), columns=[f"{n+1}orb", f"{label}{n+1}orb"]) for n, d in enumerate(data)]
    df = pd.concat(dfs, axis=1)
    for n in range(4):
        df[f"{label}{n+1}orb"] = pd.to_numeric(df[f"{label}{n+1}orb"])
    total = np.real(np.sum(tensor))/denom
    return df, total, dropped_imag


In [6]:
#Three-centre bond loops = orbital-resolved Berry curvature (bond_index_benzene_chern, index fix)
def bond_element_factory(tb, pos, lat, mu=MU):
    """rho_ab(R) = <a,0|P|b,R> from TBLR's k-resolved 1pRDM."""
    Pk = tb.solve_1pRDM_kresolved(mu)
    r0 = pos[np.newaxis, :] - pos[:, np.newaxis]                # r0[a,b] = r_b - r_a
    cache = {}
    def bond_element(R):
        key = (int(R[0]), int(R[1]))
        if key not in cache:
            r = r0 + np.asarray(R) @ lat
            ph = np.exp(-1j*np.einsum("kx,abx->kab", tb.kSpan, r))
            cache[key] = ((Pk*ph).mean(0), r)
        return cache[key]
    return bond_element

def loop_table(tb, model, pos, lat, occ, rmax=RMAX, mu=MU, tol=1e-14):
    mol = is_molecule(model)
    A_cell = 3*np.sqrt(3)/2 if mol else abs(np.linalg.det(lat))    # honeycomb cell in a^2
    bond_element = bond_element_factory(tb, pos, lat, mu)
    R_list = [[0, 0]] if mol else [[i, j] for i in range(-rmax, rmax+1) for j in range(-rmax, rmax+1)]
    cols = ["a", "b", "c", "R1x", "R1y", "R2x", "R2y", "dist_ab_x", "dist_ab_y", "dist_ca_x", "dist_ca_y",
            "oriented_area", "dist_part", "BI_real", "BI_imag", "BI_abs", "Omega_per_e", "C_imag"]
    chunks = {c: [] for c in cols}; W_total = 0.0
    for R1 in tqdm(R_list, disable=mol):
        for R2 in R_list:
            R1, R2 = np.asarray(R1), np.asarray(R2)
            rho_ab, dist_ab = bond_element(R1)            # dist_ab[a,b] = r_b - r_a  (b in cell R1)
            rho_bc, _ = bond_element(R2)
            rho_ca, dist_ca = bond_element(-R1-R2)        # dist_ca[c,a] = r_a - r_c  (c in cell R1+R2)
            BI = np.einsum("ab,bc,ca->abc", rho_ab, rho_bc, rho_ca)
            x_ab, y_ab = -dist_ab[:, :, 0], -dist_ab[:, :, 1]      # [a,b]: r_a - r_b
            x_ac, y_ac = dist_ca[:, :, 0].T, dist_ca[:, :, 1].T    # [a,c]: r_a - r_c
            dist_part = np.einsum("ab,ac->abc", x_ab, y_ac) - np.einsum("ab,ac->abc", y_ab, x_ac)
            W = dist_part*BI.imag
            W_total += W.sum()
            m = np.abs(W) > tol
            if not m.any():
                continue
            ai, bi, ci = np.where(m); nm = len(ai)
            vals = dict(a=ai, b=bi, c=ci, R1x=np.full(nm, R1[0]), R1y=np.full(nm, R1[1]),
                        R2x=np.full(nm, R2[0]), R2y=np.full(nm, R2[1]),
                        dist_ab_x=x_ab[ai, bi], dist_ab_y=y_ab[ai, bi], dist_ca_x=x_ac[ai, ci], dist_ca_y=y_ac[ai, ci],
                        oriented_area=0.5*dist_part[m], dist_part=dist_part[m],
                        BI_real=BI.real[m], BI_imag=BI.imag[m], BI_abs=np.abs(BI[m]),
                        Omega_per_e=W[m]/occ,                     # contribution to Omega^xy per electron (a^2)
                        C_imag=2*np.pi/A_cell*W[m])               # contribution to C (bulk) / Fig. 4 loop weight (molecules)
            for c in cols:
                chunks[c].append(vals[c])
    df = pd.DataFrame({c: (np.concatenate(chunks[c]) if chunks[c] else np.array([])) for c in cols})
    df.attrs["C_all_loops"] = float(2*np.pi/A_cell*W_total)      # before the tol cut
    return df, A_cell, bond_element

def loop_families(dfl):
    if not len(dfl):
        return pd.DataFrame(columns=["sides_over_a", "n_loops", "Omega_per_e", "C_imag", "sum_abs_C", "fraction_of_total"])
    ab = np.hypot(dfl.dist_ab_x, dfl.dist_ab_y); ca = np.hypot(dfl.dist_ca_x, dfl.dist_ca_y)
    bc = np.hypot(dfl.dist_ab_x-dfl.dist_ca_x, dfl.dist_ab_y-dfl.dist_ca_y)
    key = [", ".join(f"{x:.3f}" for x in sorted(t)) for t in zip(ab, bc, ca)]
    g = dfl.assign(sides_over_a=key).groupby("sides_over_a").agg(
        n_loops=("C_imag", "size"), Omega_per_e=("Omega_per_e", "sum"),
        C_imag=("C_imag", "sum"), sum_abs_C=("C_imag", lambda s: s.abs().sum()))
    tot = dfl.C_imag.sum()
    g["fraction_of_total"] = g.C_imag/tot if abs(tot) > 1e-8 else np.nan
    return g.reindex(g.sum_abs_C.sort_values(ascending=False).index).reset_index()

def g_bond(bond_element, pos, lat, occ, mol, rmax=RMAX):
    """two-centre check, Eq. 1: Tr g = 1/(2 Ne) sum |rho_ab(R)|^2 |r_b + R - r_a|^2"""
    Rs = [[0, 0]] if mol else [[i, j] for i in range(-2*rmax, 2*rmax+1) for j in range(-2*rmax, 2*rmax+1)]
    return sum((np.abs(r)**2*(d**2).sum(-1)).sum() for r, d in (bond_element(R) for R in Rs))/(2*occ)

In [7]:
#Run: TBLR object per model, QGT tensors, loops, and write everything
summary = []
import time
for name, my_model in build_models().items():
    t0 = time.time()
    mol = is_molecule(my_model)
    H, lat, pos = hopping_table(my_model)
    ham_func = make_ham_func(H, my_model)
    sc = name.startswith("haldane_")
    nk = 1 if mol else (NK//SC if sc else NK)
    rmax = int(np.ceil(RMAX/SC)) if sc else RMAX
    tb = TBLR.TBLinearResponse(ham_func, Lar=np.array(lat),
                               Nkp=[nk, nk],
                               prestr=f"{name}_tp{T2_MAG}",
                               save_all=False,
                               kpath=['G', 'X', 'M', 'G', 'Y', 'M'])
    occ = int(np.sum(tb.enUM[0, :, 0].real <= MU))
    print(f"{name}: norb={tb.norb}  occ={occ}  k-points={len(tb.kSpan)}")

    qgt = calc_QM(tb, occ)
    atoms = [[i] for i in range(tb.norb)]       # one orbital per site (Haldane cell: the two sublattices)
    QM_tensor = qgt["xx"] + qgt["yy"]            # metric trace, as in the original cell
    BC_tensor = 1j*(qgt["xy"] - qgt["yx"])       # Berry curvature Omega^xy; its total is -2 Im Q^xy
    df_qm, tot_qm, im_qm = fluctuation_table(QM_tensor, atoms, "QM")
    df_bc, tot_bc, im_bc = fluctuation_table(BC_tensor, atoms, "BC")
    dfl, A_cell, bond_element = loop_table(tb, my_model, pos, lat, occ, rmax=rmax, tol=LOOP_TOL_SC if sc else (LOOP_TOL if not mol else 1e-14))

    d = os.path.join(OUT, name); os.makedirs(d, exist_ok=True)
    for c in qgt:
        np.save(os.path.join(d, f"qgt_{c}.npy"), qgt[c])
    df_qm.to_csv(os.path.join(d, "fluctuation_quantum_metric.csv"), index=False)
    df_bc.to_csv(os.path.join(d, "fluctuation_berry_curvature.csv"), index=False)
    dfl.to_csv(os.path.join(d, "berry_loops.csv"), index=False)
    loop_families(dfl).to_csv(os.path.join(d, "berry_loop_families.csv"), index=False)

    # cross-checks: TBLR's own k-resolved metric, the two-centre bond formula, and the two Berry partitions
    gxx_tblr = np.sum(tb.solve_QGT_kresolved(mu=MU, dkx=np.array([1, 0]), k_step=K_STEP)).real/(len(tb.kSpan)*occ)
    gyy_tblr = np.sum(tb.solve_QGT_kresolved(mu=MU, dkx=np.array([0, 1]), k_step=K_STEP)).real/(len(tb.kSpan)*occ)
    Om4 = tot_bc
    print(f"   {name} done in {time.time()-t0:.0f} s", flush=True)
    summary.append(dict(system=name, norb=tb.norb, N_e=occ, k_points=len(tb.kSpan), molecule=mol,
        Tr_g_4index=tot_qm, QM_fluct_sum=sum(df_qm[f"QM{n}orb"].sum() for n in range(1, 5)),
        BC_fluct_sum=sum(df_bc[f"BC{n}orb"].sum() for n in range(1, 5)), max_dropped_imag=max(im_qm, im_bc), Tr_g_TBLR=gxx_tblr+gyy_tblr,
        Tr_g_bond_Eq1=g_bond(bond_element, pos, lat, occ, mol, rmax=rmax),
        Omega_4index=Om4, Omega_loops=float(dfl.Omega_per_e.sum()) if len(dfl) else 0.0,
        C_4index=0.0 if mol else 2*np.pi*occ/A_cell*Om4,
        C_loops=float(dfl.C_imag.sum()) if len(dfl) else 0.0, C_loops_before_cut=dfl.attrs.get("C_all_loops", 0.0),
        sum_abs_C_loops=float(dfl.C_imag.abs().sum()) if len(dfl) else 0.0,
        n_loop_rows=len(dfl), rmax=None if mol else rmax))
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT, "summary.csv"), index=False)
json.dump(dict(T1=T1, T2_MAG=T2_MAG, PHI=PHI, DELTA=DELTA, NK=NK, RMAX=RMAX, MU=MU, BETA=BETA, K_STEP=K_STEP,
               length_unit="a (NN bond)", energy_unit="|t|",
               qgt_index_order="Q[a,b,c,d] = <o|v_ad|u><u|v_bc|o>/(E_u-E_o)^2, per electron"),
          open(os.path.join(OUT, "parameters.json"), "w"), indent=2)
pd.set_option("display.float_format", lambda x: f"{x:.6g}")
summary

haldane: norb=2  occ=1  k-points=10000


  0%|          | 0/10000 [00:00<?, ?it/s]

  1%|          | 64/10000 [00:00<00:15, 639.74it/s]

  1%|▏         | 132/10000 [00:00<00:14, 660.71it/s]

  2%|▏         | 199/10000 [00:00<00:18, 517.25it/s]

...

coronene: norb=24  occ=12  k-points=1


/home/claude/qgt_tblr/TBLR2.py:839: RuntimeWarning: divide by zero encountered in divide
  result = np.where(np.abs(emn) < delta, 0, fnm / (emn**2))
/home/claude/qgt_tblr/TBLR2.py:839: RuntimeWarning: invalid value encountered in divide
  result = np.where(np.abs(emn) < delta, 0, fnm / (emn**2))


   coronene done in 11 s


/home/claude/qgt_tblr/TBLR2.py:839: RuntimeWarning: divide by zero encountered in divide
  result = np.where(np.abs(emn) < delta, 0, fnm / (emn**2))
/home/claude/qgt_tblr/TBLR2.py:839: RuntimeWarning: invalid value encountered in divide
  result = np.where(np.abs(emn) < delta, 0, fnm / (emn**2))


,system,norb,N_e,k_points,molecule,Tr_g_4index,QM_fluct_sum,BC_fluct_sum,max_dropped_imag,Tr_g_TBLR,Tr_g_bond_Eq1,Omega_4index,Omega_loops,C_4index,C_loops,C_loops_before_cut,sum_abs_C_loops,n_loop_rows,rmax
0,haldane,2,1,10000,False,0.860094,0.860094,0.413497,1.12537e-14,0.860094,0.860085,0.413497,0.412463,1,0.997501,0.997785,2.6368,660852,12
1,haldane_4x4,32,16,625,False,0.860094,0.860094,0.413497,1.55786e-17,0.860094,0.860089,0.413497,0.375305,1,0.907636,0.99876,2.06316,170688,3
2,benzene,6,3,1,True,0.666667,0.666667,-1.44341e-09,8.03958e-15,0.666667,0.666667,-1.44341e-09,-1.19262e-16,0,-8.25728e-16,-8.72613e-16,4.8368,120,NaN
3,coronene,24,12,1,True,0.690657,0.690657,1.87781e-10,3.03413e-15,0.690657,0.690657,1.87781e-10,-1.94289e-16,0,-6.05072e-15,-6.10829e-15,23.5269,10848,NaN


## Fluctuation-resolved quantum metric and Berry curvature

In [8]:
for name in summary.system:
    for f in ("fluctuation_quantum_metric.csv", "fluctuation_berry_curvature.csv"):
        print(f"--- {name}: {f}")
        display(pd.read_csv(os.path.join(OUT, name, f)))

--- haldane: fluctuation_quantum_metric.csv


,1orb,QM1orb,2orb,QM2orb,3orb,QM3orb,4orb,QM4orb
0,aaaa,0.345667,aaab,0.0192802,aabc,0,abcd,0
1,NaN,NaN,aaba,0.0192802,abac,0,NaN,NaN
2,NaN,NaN,aabb,-0.00239612,abbc,0,NaN,NaN
3,NaN,NaN,abaa,0.0192802,abca,0,NaN,NaN
4,NaN,NaN,abab,0.0940366,abcb,0,NaN,NaN
5,NaN,NaN,abba,0.345667,abcc,0,NaN,NaN
6,NaN,NaN,abbb,0.0192802,NaN,NaN,NaN,NaN


--- haldane: fluctuation_berry_curvature.csv


,1orb,BC1orb,2orb,BC2orb,3orb,BC3orb,4orb,BC4orb
0,aaaa,-6.298e-20,aaab,0.0946035,aabc,0,abcd,0
1,NaN,NaN,aaba,0.0946035,abac,0,NaN,NaN
2,NaN,NaN,aabb,-2.20657e-19,abbc,0,NaN,NaN
3,NaN,NaN,abaa,0.0946035,abca,0,NaN,NaN
4,NaN,NaN,abab,0.0350828,abcb,0,NaN,NaN
5,NaN,NaN,abba,-6.298e-20,abcc,0,NaN,NaN
6,NaN,NaN,abbb,0.0946035,NaN,NaN,NaN,NaN


--- haldane_4x4: fluctuation_quantum_metric.csv


...

,1orb,BC1orb,2orb,BC2orb,3orb,BC3orb,4orb,BC4orb
0,aaaa,0,aaab,0,aabc,-0.104336,abcd,0.240563
1,NaN,NaN,aaba,0,abac,-0.0159452,NaN,NaN
2,NaN,NaN,aabb,4.90322e-18,abbc,0,NaN,NaN
3,NaN,NaN,abaa,0,abca,0,NaN,NaN
4,NaN,NaN,abab,7.51682e-18,abcb,-0.0159452,NaN,NaN
5,NaN,NaN,abba,0,abcc,-0.104336,NaN,NaN
6,NaN,NaN,abbb,0,NaN,NaN,NaN,NaN


--- coronene: fluctuation_quantum_metric.csv


,1orb,QM1orb,2orb,QM2orb,3orb,QM3orb,4orb,QM4orb
0,aaaa,0,aaab,0,aabc,0.00875815,abcd,0.148243
1,NaN,NaN,aaba,0,abac,0.0286081,NaN,NaN
2,NaN,NaN,aabb,0.0622914,abbc,0,NaN,NaN
3,NaN,NaN,abaa,0,abca,0,NaN,NaN
4,NaN,NaN,abab,0.40539,abcb,0.0286081,NaN,NaN
5,NaN,NaN,abba,0,abcc,0.00875815,NaN,NaN
6,NaN,NaN,abbb,0,NaN,NaN,NaN,NaN


--- coronene: fluctuation_berry_curvature.csv


,1orb,BC1orb,2orb,BC2orb,3orb,BC3orb,4orb,BC4orb
0,aaaa,0,aaab,0,aabc,0.0010339,abcd,-0.111581
1,NaN,NaN,aaba,0,abac,0.0547564,NaN,NaN
2,NaN,NaN,aabb,4.2338e-19,abbc,0,NaN,NaN
3,NaN,NaN,abaa,0,abca,0,NaN,NaN
4,NaN,NaN,abab,-2.00147e-19,abcb,0.0547564,NaN,NaN
5,NaN,NaN,abba,0,abcc,0.0010339,NaN,NaN
6,NaN,NaN,abbb,0,NaN,NaN,NaN,NaN


## Leading loop families (Berry curvature, three-centre partition)

In [9]:
for name in summary.system:
    fam = pd.read_csv(os.path.join(OUT, name, "berry_loop_families.csv"))
    print(f"--- {name}: {len(fam)} families  (n_loops counts ordered terms: 3 anchors x 2 orientations per triangle)")
    display(fam.head(6))

--- haldane: 8896 families  (n_loops counts ordered terms: 3 anchors x 2 orientations per triangle)


,sides_over_a,n_loops,Omega_per_e,C_imag,sum_abs_C,fraction_of_total
0,"1.000, 1.000, 1.732",36,0.172256,0.416585,0.416585,0.417628
1,"1.732, 1.732, 1.732",24,0,0,0.387968,0
2,"1.000, 1.732, 2.000",72,0.0528811,0.127888,0.127888,0.128208
3,"1.732, 3.464, 4.583",144,0,0,0.107461,0
4,"1.732, 4.000, 4.359",72,0.0334254,0.080836,0.080836,0.0810385
5,"1.732, 4.583, 5.196",144,0,0,0.0650503,0


--- haldane_4x4: 131 families  (n_loops counts ordered terms: 3 anchors x 2 orientations per triangle)


,sides_over_a,n_loops,Omega_per_e,C_imag,sum_abs_C,fraction_of_total
0,"1.000, 1.000, 1.732",576,0.172256,0.416585,0.416585,0.458977
1,"1.732, 1.732, 1.732",384,-4.33681e-19,-1.0842e-18,0.387968,-1.19453e-18
2,"1.000, 1.732, 2.000",1152,0.0528811,0.127888,0.127888,0.140902
3,"1.732, 3.464, 4.583",2304,4.06576e-20,2.71051e-20,0.107461,2.98633e-20
4,"1.732, 4.000, 4.359",1152,0.0334254,0.080836,0.080836,0.0890621
5,"1.732, 4.583, 5.196",2304,3.89635e-20,1.25361e-19,0.0650503,1.38118e-19


--- benzene: 3 families  (n_loops counts ordered terms: 3 anchors x 2 orientations per triangle)


,sides_over_a,n_loops,Omega_per_e,C_imag,sum_abs_C,fraction_of_total
0,"1.000, 1.732, 2.000",72,-0.333333,-2.4184,2.4184,NaN
1,"1.732, 1.732, 1.732",12,0.25,1.8138,1.8138,NaN
2,"1.000, 1.000, 1.732",36,0.0833333,0.6046,0.6046,NaN


--- coronene: 73 families  (n_loops counts ordered terms: 3 anchors x 2 orientations per triangle)


,sides_over_a,n_loops,Omega_per_e,C_imag,sum_abs_C,fraction_of_total
0,"1.000, 1.000, 1.732",288,0.158806,4.60869,4.60869,NaN
1,"1.732, 1.732, 1.732",156,-0.0265766,-0.771275,3.99888,NaN
2,"1.000, 1.732, 2.000",504,-0.0292953,-0.850173,2.32938,NaN
3,"1.000, 1.732, 2.646",432,-0.0456225,-1.324,1.324,NaN
4,"1.732, 3.464, 4.583",288,-0.00965161,-0.280097,0.934131,NaN
5,"1.732, 1.732, 3.000",288,0.00115155,0.0334189,0.804197,NaN
